# 07 - Computing Signal Power Two Ways

This notebook computes IQ signal power using two independent methods and verifies they match.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Generate Dummy IQ Data

In [ ]:
np.random.seed(42)
n_samples = 4096
sample_rate = 1e6
t = np.arange(n_samples) / sample_rate

iq = (1.0 + 0.5j) * np.exp(2j * np.pi * 100e3 * t) + 0.1 * (np.random.randn(n_samples) + 1j * np.random.randn(n_samples))
print(f"IQ shape: {iq.shape}, dtype: {iq.dtype}")

## Method 1: Direct Complex Magnitude

Signal power = mean(|IQ|^2)

In [ ]:
power_method1 = np.mean(np.abs(iq) ** 2)
print(f"Method 1 (|IQ|^2): {power_method1:.6f}")

## Method 2: I/Q Component Decomposition

Signal power = mean(I^2 + Q^2)

In [ ]:
I = iq.real
Q = iq.imag
power_method2 = np.mean(I ** 2 + Q ** 2)
print(f"Method 2 (I^2 + Q^2): {power_method2:.6f}")

## Verification

In [ ]:
print(f"Method 1: {power_method1:.10f}")
print(f"Method 2: {power_method2:.10f}")
print(f"Difference: {abs(power_method1 - power_method2):.2e}")
print(f"Match (np.isclose): {np.isclose(power_method1, power_method2)}")

## Floating-Point Discrepancies

Both methods should produce identical results mathematically:
$$|IQ|^2 = I^2 + Q^2$$

However, floating-point arithmetic can introduce tiny differences due to operation ordering. These differences are typically at machine epsilon level (~1e-7 for float64).

In [ ]:
# Show floating-point behavior with float32
iq_f32 = iq.astype(np.complex64)
p1_f32 = np.mean(np.abs(iq_f32) ** 2)
p2_f32 = np.mean(iq_f32.real ** 2 + iq_f32.imag ** 2)
print(f"float32 - Method 1: {p1_f32:.10f}")
print(f"float32 - Method 2: {p2_f32:.10f}")
print(f"float32 - Difference: {abs(p1_f32 - p2_f32):.2e}")
print(f"float32 - Match: {np.isclose(p1_f32, p2_f32)}")

## Physical Meaning of Signal Power

Signal power represents the average energy per sample. In RF/signal processing:
- Power is proportional to the squared magnitude of the complex envelope
- Consistency between methods validates your data pipeline
- Power measurements are used for SNR estimation, gain control, and signal detection

## Exercise: Sliding Window Power

Compute signal power over a sliding window of 256 samples and plot power vs. time.

In [ ]:
# YOUR CODE HERE
pass

<details>
<summary>Solution</summary>

```python
window_size = 256
n_windows = n_samples - window_size + 1
power_windows = np.array([np.mean(np.abs(iq[i:i+window_size])**2) for i in range(n_windows)])
t_windows = np.arange(window_size//2, window_size//2 + n_windows) / sample_rate

plt.figure(figsize=(12, 4))
plt.plot(t_windows * 1e6, 10 * np.log10(power_windows))
plt.xlabel('Time (us)')
plt.ylabel('Power (dB)')
plt.title('Signal Power vs. Time')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
```
</details>

## Summary

- Method 1: `mean(|IQ|^2)` — operates on complex magnitude directly
- Method 2: `mean(I^2 + Q^2)` — decomposes into real/imaginary parts
- Both methods are mathematically equivalent and should produce matching results
- Consistency checks between independent implementations help catch bugs early
- Sliding window power reveals time-varying signal characteristics